In [1]:
import pandas as pd
import numpy as np


df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

df.fillna({'Age': df['Age'].median()}, inplace=True)
df.dropna(subset=['Embarked'], inplace=True)
df['Title'] = df['Name'].str.extract(r', ([A-Za-z]+)\.')
df['AgeGroup'] = df['Age'].apply(lambda x: 'Child' if x<13 else 'Teen' if x<18 else 'Adult' if x<61 else 'Senior')
df['Sex_encoded'] = df['Sex'].map({'male':0, 'female':1})

In [2]:
features = ['Pclass', 'Age', 'Fare', 'Sex_encoded']
X = df[features]
y = df['Survived']

In [3]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score

bst = XGBClassifier()

cross_val = cross_val_score(bst, X,y, cv=5)
print(cross_val)
print(np.mean(cross_val))



[0.78651685 0.81460674 0.85955056 0.7752809  0.84180791]
0.815552593156859


# **week4/ day 5**

1.	Hyperparameter tuning: GridSearchCV vs RandomizedSearchCV
2. Optuna: Bayesian optimisation, run a 50-trial study on XGBoost
3.	Compare: grid search (exhaustive) vs Optuna (smart): same budget, different results


## Task 1: GridSearchCV
1. Define a param grid with n_estimators, max_depth, learning_rate (3 values each = 27 combinations).
2. Run on XGBoost with Titanic data. Print best params and best score.


In [6]:
from sklearn.model_selection import GridSearchCV

bst = XGBClassifier()

clf = GridSearchCV(estimator=bst,
             param_grid={
                 'n_estimators':[100,200,500],
                 'max_depth':[3,6,9],
                 "learning_rate": [0.01, 0.1, 0.3]
             },cv=5)


In [7]:
clf.fit(X,y)


GridSearchCV(cv=5,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False, eval_metric=None,
                                     feature_types=None, feature_weights=None,
                                     gamma=None, grow_policy=None,
                                     importance_type=None,
                                     interaction_constraints=None,
                                     learning_rate=None, max_bin=None,
                                     max_cat_threshold=None,
                                     max_cat_to_onehot=None,
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=None, num_parallel_tree=None, ...),
             param_grid={'learning_rate': [0.01, 0.1, 0.3],
                         'max_depth': [3, 6, 9],
                         'n_estimators': [100, 200, 500]})

In [8]:
clf.best_params_

{'learning_rate': 0.01, 'max_depth': 6, 'n_estimators': 500}

In [9]:
clf.best_score_

np.float64(0.8391544467720434)

## GridSearchCV
1. Best_score- = 83.9 %, but 27 combinations

## Task 2  RandomizedSearchCV
1. Same param grid, n_iter=10 (only 10 random combinations).
2. Print best params and best score. Compare to GridSearch.


In [15]:
from sklearn.model_selection import RandomizedSearchCV
param_grid = {
    'n_estimators':[100,200,500],
    'max_depth':[3,6,9],
    "learning_rate": [0.01, 0.1, 0.3],
}
rnd_CV = RandomizedSearchCV(bst, param_grid,cv=5,n_iter=10)
rnd_CV.fit(X,y)
print(rnd_CV.best_score_)
print(rnd_CV.best_params_)


0.8391544467720434
{'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.01}


## RandomizedSearchCV
1. Best_score- = 83.9 % and same params as GridSearchCV but only 10 combinations

## Task 3 Optuna
1. Install optuna, run a 50-trial study on XGBoost.
2. Define an objective function that suggests params and returns CV score.
3. Print best params and best score.


In [16]:
!pip install optuna -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 14.2 MB/s eta 0:00:00


In [17]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

1. For optuna we write an OBJECTIVE function
2. Then optuna calls this function for 50 times:
      1. Each time suggesting  different params and to maximise return value

In [20]:
def objective(trial):
  #optune suggests trial
  n_estimators = trial.suggest_int('n_estimators', 100, 500)
  max_depth = trial.suggest_int('max_depth', 3, 9)
  learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)

  #define model with suggested params
  model = XGBClassifier(
      n_estimators = n_estimators,
      max_depth = max_depth,
      learning_rate = learning_rate
  )
  score= cross_val_score(model, X, y, cv=5)
  return np.mean(score)

In [21]:
#50 - trial study
study = optuna.create_study(direction="maximize")
study.optimize(objective,n_trials=50)

In [23]:
print(study.best_params)
print(study.best_value)

{'n_estimators': 129, 'max_depth': 3, 'learning_rate': 0.1349177716495117}
0.8458960198057512


##Task 4  Comparison markdown



### GridSearchCV vs RandomizedSearchCV vs Optuna

| Method | Combinations tried | Best score |
|--------|-------------------|------------|
| GridSearchCV | 27 (exhaustive) | 0.839 |
| RandomizedSearchCV | 10 (random) | 0.839 |
| Optuna | 50 (smart) | 0.846 |

## Task 5 : When to Use Each in a Real Project

**GridSearchCV: **
1.Use when search space is small (< 100 combinations) and
compute time is not a concern.
2. Guarantees finding the best combination
within the defined grid.
3. Good for final fine-tuning around known good params.

**RandomizedSearchCV:**
1. Use when search space is large and you need a quick
baseline.
2. Samples randomly: no learning between trials.
3. Good for initial exploration before committing to a full search.

**Optuna:**
1. Best default choice for most real projects.
2. Learns from previous trials using TPE (Tree-structured Parzen Estimator) to focus on promising parameter regions.
3. Found 0.846 vs GridSearch 0.839 in the same budget:smarter search, better results.

**Rule of thumb: **
1. Start with Optuna.
2. Use GridSearch only for small final fine-tuning around Optuna's best params.

## Hyperparameter Optimisation
1. Finds the best settings for the model before training
2. Examples: GridSearch, RandomizedSearch, Optuna